# Stage 0 Load Datasets

In [1]:
import pandas as pd
path  = 'Data' # Path of the dataset folder
task_id = 18800 # Choose the task
df_train = pd.read_csv(f"{path}/task_{task_id}/df_{task_id}_train.csv")
print("data is loaded...")

data is loaded...


In [2]:
from Dataset_Utility import utility_functions as uf
uf.calculate_label_rate(df_train)

Total Sample size is 5367, Positive Sample size is 159, Negative Sample size is 5208, label rate is 0.03


# Stage 1 Synthetic Data Generation

CTAB-GAN and Tabula need to be downloaded from the GitHub link.

In [ ]:
path_model = 'Generative_Models' # Path of the folder containing CTAB-GAN and Tabula

## Stage 1.1 Generate Data Using CTGAN

https://github.com/sdv-dev/CTGAN

In [ ]:
#!pip install ctgan

In [4]:
from ctgan import CTGAN

discrete_columns = df_train.columns
ctgan = CTGAN()
ctgan.fit(df_train, discrete_columns)
ctgan_sample = ctgan.sample(df_train.shape[0])

uf.calculate_label_rate(ctgan_sample)

ctgan_sample.to_csv(f"{path}/task_{task_id}/df_{task_id}_syn_ctgan.csv", index = False)

Total Sample size is 5367, Positive Sample size is 360, Negative Sample size is 5007, label rate is 0.07


## Stage 1.2 Generate Data Using CTAB-GAN

https://github.com/Team-TUD/CTAB-GAN

In [ ]:
import sys
sys.path.append(f'{path_model}/CTAB-GAN')
from model.ctabgan import CTABGAN

real_path = f"{path}/task_{task_id}/df_{task_id}_train.csv" # Path of the training data
ctabgan =  CTABGAN(raw_csv_path = real_path,
                 test_ratio = 0.20,
                 categorical_columns = ['label'],
                 log_columns = [],
                 integer_columns = ['age', 'gender', 'residence', 'city', 'city_rank','series_dev', 'series_group', 'emui_dev', 'device_name', 'device_size','creat_type_cd', 'slot_id', 'u_refreshTimes', 'u_feedLifeCycle'],
                 problem_type= {"Classification": 'label'},
                 epochs = 50)
ctabgan.fit()
ctabgan_sample = ctabgan.generate_samples()

# Label in the generated dataset is of "object" type. Convert it to "int64"
ctabgan_sample['label'] = ctabgan_sample['label'].astype("int64")

uf.calculate_label_rate(ctabgan_sample)

ctabgan_sample.to_csv(f"{path}/task_{task_id}/df_{task_id}_syn_ctabgan.csv", index = False)

## Stage 1.3 Generate Data Using TVAE

https://sdv.dev/SDV/user_guides/single_table/tvae.html

In [6]:
from ctgan import TVAE

tvae = TVAE()
tvae.fit(df_train, discrete_columns)
tvae_sample = tvae.sample(df_train.shape[0])

uf.calculate_label_rate(tvae_sample)

tvae_sample.to_csv(f"{path}/Data/task_{task_id}/df_{task_id}_syn_tvae.csv", index = False)

Total Sample size is 5367, Positive Sample size is 75, Negative Sample size is 5292, label rate is 0.01


## Stage 1.4 Generate Data Using DataSynthesizer

https://github.com/DataResponsibly/DataSynthesizer

In [ ]:
#!pip install DataSynthesizer

In [7]:
from DataSynthesizer.DataDescriber import DataDescriber
from DataSynthesizer.DataGenerator import DataGenerator
from DataSynthesizer.ModelInspector import ModelInspector
from DataSynthesizer.lib.utils import read_json_file, display_bayesian_network

epsilon = 0

input_data = f"{path}/task_{task_id}/df_{task_id}_train.csv"
mode = 'correlated_attribute_mode'
description_file = f'{path}/task_{task_id}/description.json'
synthetic_data = f'{path}/task_{task_id}/sythetic_data.csv'

degree_of_bayesian_network = 2
num_tuples_to_generate = df_train.shape[0]

describer = DataDescriber()
describer.describe_dataset_in_correlated_attribute_mode(dataset_file=input_data,
                                                        epsilon=epsilon,
                                                        k=degree_of_bayesian_network)
describer.save_dataset_description_to_file(description_file)
generator = DataGenerator()
generator.generate_dataset_in_correlated_attribute_mode(num_tuples_to_generate, description_file)
generator.save_synthetic_data(synthetic_data)

ds_sample = pd.read_csv(synthetic_data)

uf.calculate_label_rate(ds_sample)

ds_sample.to_csv(f"{path}/task_{task_id}/df_{task_id}_syn_ds.csv", index = False)

Total Sample size is 5367, Positive Sample size is 175, Negative Sample size is 5192, label rate is 0.03


## Stage 1.5 Generate Data Using GReaT

https://github.com/kathrinse/be_great/tree/main

In [ ]:
#!pip install be-great
#!pip install transformers[torch]
#!pip install accelerate -U

In [8]:
from be_great import GReaT
import numpy as np
great = GReaT(llm='distilgpt2', batch_size = 100, epochs = 50)
great.fit(df_train)
great_sample = great.sample(n_samples=df_train.shape[0])

uf.calculate_label_rate(great_sample)

great_sample.to_csv(f"{path}/task_{task_id}/df_{task_id}_syn_great.csv", index = False)

Total Sample size is 5367, Positive Sample size is 42, Negative Sample size is 5325, label rate is 0.01


## Stage 1.6 Generate Data Using Tabula

https://github.com/zhao-zilong/Tabula

In [ ]:
import sys
sys.path.append(f'{path_model}/Tabula')
from tabula_middle_padding import Tabula
import torch

tabula = Tabula(llm='distilgpt2', experiment_dir = "df_train", batch_size=100, epochs=20)
tabula.fit(df_train, conditional_col = df_train.columns[0])
torch.save(tabula.model.state_dict(), "df_train.pt")
tabula_sample = tabula.sample(n_samples=df_train.shape[0])

uf.calculate_label_rate(tabula_sample)

tabula_sample.to_csv(f"{path}/task_{task_id}/df_{task_id}_syn_tabula.csv", index = False)